<a href="https://colab.research.google.com/github/MaGiaVy/DoAnPython/blob/main/project/notebooks/Nhom3thangcuti_Tuan4/Tuan4_GiaVy_Pipeline%2BpHash_BACKUP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏆 Pipeline truy xuất ảnh sản phẩm: CLIP/MobileCLIP + FAISS + pHash + DINOv2 Re-ranking

**Bản đã sửa để chạy ổn định trên VS Code / Windows.**

**Giữ lại thuật toán để đảm bảo tính trung thực:**
- CLIP HuggingFace mặc định, Apple MobileCLIP vẫn giữ dưới dạng tùy chọn.
- Image + Text feature fusion với alpha tuning trên validation set.
- FAISS IndexFlatIP để truy xuất nhanh, không làm tăng điểm giả.
- pHash/aHash chỉ dùng boost nhẹ ảnh gần trùng, không thay thế mô hình chính.
- DINOv2-Small dùng re-ranking ở giai đoạn 2.
- Test set chỉ dùng sau khi đã tuning xong trên validation set.


## ⚙️ Cell 0: Kiểm tra GPU & Runtime

In [1]:
# ══════════════════════════════════════════════════════════════════════
# CELL 0 — CONFIG TỔNG CHO VS CODE / WINDOWS
# Giữ đủ thuật toán: CLIP/MobileCLIP fallback, FAISS, pHash, DINOv2 re-ranking.
# Không hard-code kết quả. Val dùng để tuning, Test chỉ dùng đánh giá cuối.
# ══════════════════════════════════════════════════════════════════════
import os
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# ─── Reproducibility ────────────────────────────────────────────────
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# ─── Device ─────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ─── Batch size: CPU nhỏ hơn để tránh treo máy, GPU lớn hơn một chút ──
BATCH_SIZE = 16 if DEVICE == 'cuda' else 8
NUM_WORKERS = 0  # Windows notebook nên để 0 cho ổn định

# ─── Bật / tắt thuật toán ──────────────────────────────────────────
# Mặc định KHÔNG bật MobileCLIP Apple vì trên Windows dễ lỗi dependency.
# Thuật toán vẫn giữ lại: muốn thử thì đổi PREFER_MOBILECLIP = True.
PREFER_MOBILECLIP = False
USE_MOBILECLIP = False
USE_DINOV2 = True
USE_PHASH = True
USE_TFIDF = True
USE_FAISS = True

# ─── Model config ───────────────────────────────────────────────────
MOBILECLIP_VARIANT = 'mobileclip_s0'
MOBILECLIP_CKPT = Path('models') / 'mobileclip_s0.pt'
HF_CLIP_MODEL = 'openai/clip-vit-base-patch32'

# DINOv2-Small: đúng 384 chiều, nhẹ hơn vitb14 nên ổn định hơn trên máy cá nhân.
DINO_MODEL_NAME = 'dinov2_vits14'
DINO_EXPECTED_DIM = 384

print(f'✅ Python       : {sys.version.split()[0]}')
print(f'✅ PyTorch      : {torch.__version__}')
print(f'✅ DEVICE       : {DEVICE}')
print(f'✅ BATCH_SIZE   : {BATCH_SIZE}')
print(f'✅ MobileCLIP   : {"try Apple MobileCLIP" if PREFER_MOBILECLIP else "OFF, dùng CLIP HuggingFace fallback"}')
print(f'✅ DINOv2       : {DINO_MODEL_NAME}')

✅ Python       : 3.12.10
✅ PyTorch      : 2.12.0+cu126
✅ DEVICE       : cuda
✅ BATCH_SIZE   : 16
✅ MobileCLIP   : OFF, dùng CLIP HuggingFace fallback
✅ DINOv2       : dinov2_vits14


In [2]:
# CELL 0.1 — Kiểm tra GPU an toàn cho Windows/VS Code
import subprocess
import torch

try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=10)
    if result.returncode == 0:
        print('✅ GPU khả dụng qua nvidia-smi:')
        print(result.stdout[:2000])
    else:
        print('ℹ️ Không thấy nvidia-smi. Nếu torch.cuda=False thì đang chạy CPU.')
except Exception as e:
    print(f'ℹ️ Bỏ qua nvidia-smi: {e}')

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name      : {torch.cuda.get_device_name(0)}')

✅ GPU khả dụng qua nvidia-smi:
Fri Jun  5 19:09:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.86                 Driver Version: 591.86         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060      WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   56C    P0            N/A  /  115W |    1320MiB /   8188MiB |      1%      Default |
|                                         |                        |                  N/A |
+----------------

## 📦 Cell 1: Cài đặt thư viện & MobileCLIP

In [3]:
# CELL 1 — Kiểm tra thư viện cần thiết
# Chạy cell này nếu máy thiếu thư viện. Không cài faiss-gpu trên Windows, dùng faiss-cpu cho ổn định.
import importlib.util
import subprocess
import sys

packages = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'PIL': 'pillow',
    'tqdm': 'tqdm',
    'sklearn': 'scikit-learn',
    'faiss': 'faiss-cpu',
    'imagehash': 'imagehash',
    'transformers': 'transformers',
    'torchvision': 'torchvision',
    'timm': 'timm',
}

missing = [pip_name for import_name, pip_name in packages.items()
           if importlib.util.find_spec(import_name) is None]

if missing:
    print('📦 Thiếu thư viện, đang cài:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
else:
    print('✅ Các thư viện chính đã có sẵn.')

print('ℹ️ Nếu muốn thử MobileCLIP Apple thật, cài riêng:')
print('   python -m pip install git+https://github.com/apple/ml-mobileclip.git')

✅ Các thư viện chính đã có sẵn.
ℹ️ Nếu muốn thử MobileCLIP Apple thật, cài riêng:
   python -m pip install git+https://github.com/apple/ml-mobileclip.git


## 📂 Cell 2: Kết nối Google Drive & Tự động dò tìm đường dẫn Dataset

In [4]:
# CELL 2 — Đường dẫn dataset trên VS Code / Windows
from pathlib import Path
import os

DATA_DIR = Path(r'E:\study\DuLieuPython')
CSV_PATH = DATA_DIR / 'train.csv'
IMG_DIR = DATA_DIR / 'train_images'
FEAT_DIR = DATA_DIR / 'features'
FEAT_DIR.mkdir(parents=True, exist_ok=True)

print('📁 DATA_DIR:', DATA_DIR)
print('📄 CSV_PATH:', CSV_PATH)
print('🖼️ IMG_DIR :', IMG_DIR)
print('💾 FEAT_DIR:', FEAT_DIR)

if not DATA_DIR.exists():
    raise FileNotFoundError(f'❌ Không tìm thấy thư mục dữ liệu: {DATA_DIR}')
if not CSV_PATH.exists():
    raise FileNotFoundError(f'❌ Không tìm thấy train.csv tại: {CSV_PATH}')
if not IMG_DIR.exists():
    raise FileNotFoundError(f'❌ Không tìm thấy thư mục ảnh train_images tại: {IMG_DIR}')

num_images = len(list(IMG_DIR.glob('*')))
print('✅ Dữ liệu đã sẵn sàng!')
print(f'   Số file trong train_images: {num_images:,}')

📁 DATA_DIR: E:\study\DuLieuPython
📄 CSV_PATH: E:\study\DuLieuPython\train.csv
🖼️ IMG_DIR : E:\study\DuLieuPython\train_images
💾 FEAT_DIR: E:\study\DuLieuPython\features
✅ Dữ liệu đã sẵn sàng!
   Số file trong train_images: 25,850


In [5]:
# CELL 2B — Bỏ qua cell Google Colab khi chạy VS Code
print('⏭️ Đang chạy VS Code/local Windows nên bỏ qua Google Drive mount.')
print('   Dữ liệu đang dùng:', DATA_DIR)

⏭️ Đang chạy VS Code/local Windows nên bỏ qua Google Drive mount.
   Dữ liệu đang dùng: E:\study\DuLieuPython


## 🔧 Cell 3: Import & Cấu hình

In [6]:
# CELL 3 — Import & cấu hình runtime
import os
import gc
import math
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

import faiss

print(f'✅ Thiết bị      : {DEVICE}')
print(f'✅ Batch size    : {BATCH_SIZE}')
print(f'✅ Feature cache : {FEAT_DIR}')

✅ Thiết bị      : cuda
✅ Batch size    : 16
✅ Feature cache : E:\study\DuLieuPython\features


## 📊 Cell 4: Đọc dữ liệu & Chia tập (STRICT SPLIT)

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(CSV_PATH)
print(f'📊 Tổng số mẫu : {len(df):,}')
print(f'📋 Các cột     : {list(df.columns)}')
print(df.head(3))

# ─── KHÔNG stratify vì số lớp (11,014) > kích thước validation (6,850) ───
df_gallery = df.copy()

val_idx, test_idx = train_test_split(
    df.index.tolist(),
    test_size    = 0.8,
    random_state = RANDOM_SEED
)

df_val  = df.loc[val_idx].reset_index(drop=True)
df_test = df.loc[test_idx].reset_index(drop=True)

print(f'\n🗂️  Gallery size  : {len(df_gallery):,} ảnh (toàn bộ dataset)')
print(f'✅ Val queries   : {len(df_val):,} ảnh  → grid search alpha')
print(f'✅ Test queries  : {len(df_test):,} ảnh  → đánh giá cuối (1 lần!)')

📊 Tổng số mẫu : 34,250
📋 Các cột     : ['posting_id', 'image', 'image_phash', 'title', 'label_group']
         posting_id                                 image       image_phash  \
0   train_129225211  0000a68812bc7e98c42888dfb1c07da0.jpg  94974f937d4c2433   
1  train_3386243561  00039780dfc94d01db8676fe789ecd05.jpg  af3f9460c2838f0f   
2  train_2288590299  000a190fdd715a2a36faed16e2c65df7.jpg  b94cb00ed3e50f78   

                                               title  label_group  
0                          Paper Bag Victoria Secret    249114794  
1  Double Tape 3M VHB 12 mm x 4,5 m ORIGINAL / DO...   2937985045  
2        Maling TTS Canned Pork Luncheon Meat 397 gr   2395904891  

🗂️  Gallery size  : 34,250 ảnh (toàn bộ dataset)
✅ Val queries   : 6,850 ảnh  → grid search alpha
✅ Test queries  : 27,400 ảnh  → đánh giá cuối (1 lần!)


## 🍎 Cell 5: Tải MobileCLIP (hoặc Fallback CLIP)

In [8]:
# CELL 5A — Cấu hình model trước khi load
# Mặc định dùng CLIP HuggingFace để chạy ổn định trên Windows.
# Muốn thử Apple MobileCLIP thì đổi PREFER_MOBILECLIP=True ở CELL 0 và cài package mobileclip.
print('✅ Config model:')
print('   PREFER_MOBILECLIP:', PREFER_MOBILECLIP)
print('   HF_CLIP_MODEL     :', HF_CLIP_MODEL)
print('   DINO_MODEL_NAME   :', DINO_MODEL_NAME)

✅ Config model:
   PREFER_MOBILECLIP: False
   HF_CLIP_MODEL     : openai/clip-vit-base-patch32
   DINO_MODEL_NAME   : dinov2_vits14


In [9]:
# CELL 5B — Load MobileCLIP nếu có, nếu không fallback sang CLIP HuggingFace
import os
from pathlib import Path

USE_MOBILECLIP = False
clip_model = None
preprocess = None
tokenizer = None

if PREFER_MOBILECLIP:
    try:
        import mobileclip
        import urllib.request

        MOBILECLIP_CKPT.parent.mkdir(parents=True, exist_ok=True)
        CKPT_URLS = {
            'mobileclip_s0': 'https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_s0.pt',
            'mobileclip_s1': 'https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_s1.pt',
            'mobileclip_s2': 'https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_s2.pt',
            'mobileclip_b' : 'https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_b.pt',
        }
        if not MOBILECLIP_CKPT.exists():
            print(f'⏬ Download checkpoint {MOBILECLIP_VARIANT}...')
            urllib.request.urlretrieve(CKPT_URLS[MOBILECLIP_VARIANT], str(MOBILECLIP_CKPT))

        print(f'⏳ Loading Apple MobileCLIP: {MOBILECLIP_VARIANT}')
        clip_model, _, preprocess = mobileclip.create_model_and_transforms(
            MOBILECLIP_VARIANT, pretrained=str(MOBILECLIP_CKPT)
        )
        tokenizer = mobileclip.get_tokenizer(MOBILECLIP_VARIANT)
        clip_model = clip_model.to(DEVICE).eval()
        with torch.no_grad():
            dummy = torch.randn(1, 3, 256, 256).to(DEVICE)
            embed_dim = int(clip_model.encode_image(dummy).shape[-1])
        USE_MOBILECLIP = True
        MODEL_LABEL = f'MobileCLIP Apple ({MOBILECLIP_VARIANT})'
    except Exception as e:
        print(f'⚠️ Không dùng được Apple MobileCLIP: {type(e).__name__}: {e}')
        print('🔄 Fallback sang CLIP HuggingFace để notebook vẫn chạy ổn định.')

if not USE_MOBILECLIP:
    from transformers import CLIPModel, CLIPProcessor
    print(f'⏳ Loading HuggingFace CLIP: {HF_CLIP_MODEL}')
    clip_model = CLIPModel.from_pretrained(HF_CLIP_MODEL).to(DEVICE).eval()
    preprocess = CLIPProcessor.from_pretrained(HF_CLIP_MODEL)
    tokenizer = None
    embed_dim = int(clip_model.config.projection_dim)
    MODEL_LABEL = f'CLIP HuggingFace ({HF_CLIP_MODEL})'

print(f'✅ MODEL_LABEL : {MODEL_LABEL}')
print(f'📐 Embedding dim: {embed_dim}')

⏳ Loading HuggingFace CLIP: openai/clip-vit-base-patch32


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

✅ MODEL_LABEL : CLIP HuggingFace (openai/clip-vit-base-patch32)
📐 Embedding dim: 512


## 🖼️📝 Cell 6: Hàm trích xuất Image & Text Features

In [10]:
# CELL 6 — Hàm trích xuất image/text features cho MobileCLIP hoặc CLIP HuggingFace
class ShopeeImageDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, return_tensor=True):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.return_tensor = return_tensor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        fname = self.df.iloc[idx]['image']
        img_path = self.img_dir / fname
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (256, 256), (128, 128, 128))
        if self.return_tensor and self.transform:
            return self.transform(img)
        return img


def _as_tensor_features(outputs):
    """Lấy tensor feature từ nhiều kiểu output khác nhau để tránh lỗi BaseModelOutputWithPooling."""
    if isinstance(outputs, torch.Tensor):
        return outputs
    if hasattr(outputs, 'image_embeds') and outputs.image_embeds is not None:
        return outputs.image_embeds
    if hasattr(outputs, 'text_embeds') and outputs.text_embeds is not None:
        return outputs.text_embeds
    if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
        return outputs.pooler_output
    if hasattr(outputs, 'last_hidden_state') and outputs.last_hidden_state is not None:
        return outputs.last_hidden_state[:, 0]
    raise TypeError(f'Không nhận dạng được output type: {type(outputs)}')


def _l2_normalize_torch(x):
    return x / (x.norm(dim=-1, keepdim=True) + 1e-10)


def _l2_normalize_numpy(x):
    x = np.asarray(x, dtype=np.float32)
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-10)


@torch.no_grad()
def extract_image_features_clip(df_input, img_dir, batch_size=32, num_workers=0):
    all_feats = []

    if USE_MOBILECLIP:
        dataset = ShopeeImageDataset(df_input, img_dir, transform=preprocess, return_tensor=True)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=(DEVICE == 'cuda'))
        for imgs in tqdm(loader, desc='🖼️ MobileCLIP image features'):
            imgs = imgs.to(DEVICE)
            feats = clip_model.encode_image(imgs)
            feats = _l2_normalize_torch(feats)
            all_feats.append(feats.detach().cpu().float().numpy())
    else:
        dataset = ShopeeImageDataset(df_input, img_dir, return_tensor=False)
        for i in tqdm(range(0, len(dataset), batch_size), desc='🖼️ CLIP HF image features'):
            batch = [dataset[j] for j in range(i, min(i + batch_size, len(dataset)))]
            inputs = preprocess(images=batch, return_tensors='pt', padding=True)
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            outputs = clip_model.get_image_features(**inputs)
            feats = _as_tensor_features(outputs)
            feats = _l2_normalize_torch(feats)
            all_feats.append(feats.detach().cpu().float().numpy())

    return _l2_normalize_numpy(np.vstack(all_feats))


@torch.no_grad()
def extract_text_features_clip(df_input, batch_size=256):
    import re

    def clean_title_shopee(title):
        title = str(title).lower()
        title = re.sub(r'[^\w\s]', ' ', title)
        return ' '.join(title.split())

    titles = [clean_title_shopee(t) for t in df_input['title'].fillna('').tolist()]
    all_feats = []

    if USE_MOBILECLIP:
        for i in tqdm(range(0, len(titles), batch_size), desc='📝 MobileCLIP text features'):
            tokens = tokenizer(titles[i:i + batch_size]).to(DEVICE)
            feats = clip_model.encode_text(tokens)
            feats = _l2_normalize_torch(feats)
            all_feats.append(feats.detach().cpu().float().numpy())
    else:
        for i in tqdm(range(0, len(titles), batch_size), desc='📝 CLIP HF text features'):
            inputs = preprocess(text=titles[i:i + batch_size], return_tensors='pt',
                                padding=True, truncation=True, max_length=77)
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            outputs = clip_model.get_text_features(**inputs)
            feats = _as_tensor_features(outputs)
            feats = _l2_normalize_torch(feats)
            all_feats.append(feats.detach().cpu().float().numpy())

    return _l2_normalize_numpy(np.vstack(all_feats))

print('✅ Hàm trích xuất features đã sẵn sàng')

✅ Hàm trích xuất features đã sẵn sàng


## 🔀 Cell 7: Fusion, FAISS & Metrics Utils

In [11]:
def fuse_and_normalize_clip(img_feats, txt_feats, alpha):
    import numpy as np
    img_feats_norm = img_feats / (np.linalg.norm(img_feats, axis=1, keepdims=True) + 1e-10)
    txt_feats_norm = txt_feats / (np.linalg.norm(txt_feats, axis=1, keepdims=True) + 1e-10)
    fused = (alpha * img_feats_norm + (1 - alpha) * txt_feats_norm).astype(np.float32)
    faiss.normalize_L2(fused)
    return fused


def build_faiss_index(features):
    index = faiss.IndexFlatIP(features.shape[1])
    index.add(features)
    return index

def get_ground_truth_dict(df_input):
    gt = {}
    for _, grp in df_input.groupby('label_group'):
        ids = set(grp['posting_id'].tolist())
        for pid in ids:
            gt[pid] = ids
    return gt


def evaluate_retrieval_clip(query_df, gallery_df, query_img, query_txt,
                             gallery_img, gallery_txt, alpha, K=5):
    q_fused      = fuse_and_normalize_clip(query_img, query_txt, alpha)
    g_fused      = fuse_and_normalize_clip(gallery_img, gallery_txt, alpha)
    gt_dict      = get_ground_truth_dict(gallery_df)
    index        = build_faiss_index(g_fused)
    _, indices   = index.search(q_fused, K + 1)
    gallery_pids = gallery_df['posting_id'].tolist()

    ap_list, p1_list, r5_list = [], [], []

    for i, row in enumerate(query_df.itertuples()):
        qid      = row.posting_id
        relevant = gt_dict.get(qid, set()) - {qid}
        if not relevant:
            continue

        retrieved = []
        for idx in indices[i]:
            pid = gallery_pids[idx]
            if pid != qid:
                retrieved.append(pid)
            if len(retrieved) == K:
                break

        hits, ap = 0, 0.0
        for rank, pid in enumerate(retrieved, 1):
            if pid in relevant:
                hits += 1
                ap   += hits / rank
        ap_list.append(ap / min(len(relevant), K))
        p1_list.append(1.0 if (retrieved and retrieved[0] in relevant) else 0.0)
        r5_list.append(len(set(retrieved) & relevant) / min(len(relevant), K))

    return {
        'mAP@5'      : float(np.mean(ap_list)),
        'Precision@1': float(np.mean(p1_list)),
        'Recall@5'   : float(np.mean(r5_list)),
    }


print('✅ Hàm fusion / FAISS / evaluate đã sẵn sàng')

✅ Hàm fusion / FAISS / evaluate đã sẵn sàng


## 🔍 Cell 8: Trích xuất tất cả Features

In [12]:
# CELL 8 — Trích xuất / load cache features
# Cache giúp chạy lại notebook không phải extract từ đầu. Nếu đổi model/dataset thì xóa thư mục FEAT_DIR.
MODEL_CACHE_TAG = 'mobileclip' if USE_MOBILECLIP else 'clip_hf'


def load_or_extract_feature(cache_name, extractor):
    cache_path = FEAT_DIR / cache_name
    if cache_path.exists():
        arr = np.load(cache_path).astype(np.float32)
        print(f'✅ Load cache: {cache_path.name} | shape={arr.shape}')
        return arr
    arr = extractor().astype(np.float32)
    np.save(cache_path, arr)
    print(f'💾 Saved cache: {cache_path.name} | shape={arr.shape}')
    return arr

print('📦 Gallery image features...')
gallery_img_feats = load_or_extract_feature(
    f'{MODEL_CACHE_TAG}_gallery_img.npy',
    lambda: extract_image_features_clip(df_gallery, IMG_DIR, BATCH_SIZE, NUM_WORKERS)
)
print('📦 Gallery text features...')
gallery_txt_feats = load_or_extract_feature(
    f'{MODEL_CACHE_TAG}_gallery_txt.npy',
    lambda: extract_text_features_clip(df_gallery)
)
print(f'✅ Gallery img: {gallery_img_feats.shape} | txt: {gallery_txt_feats.shape}')

print('\n📦 Val image features...')
val_img_feats = load_or_extract_feature(
    f'{MODEL_CACHE_TAG}_val_img.npy',
    lambda: extract_image_features_clip(df_val, IMG_DIR, BATCH_SIZE, NUM_WORKERS)
)
print('📦 Val text features...')
val_txt_feats = load_or_extract_feature(
    f'{MODEL_CACHE_TAG}_val_txt.npy',
    lambda: extract_text_features_clip(df_val)
)
print(f'✅ Val img: {val_img_feats.shape} | txt: {val_txt_feats.shape}')

print('\n📦 Test image features...')
test_img_feats = load_or_extract_feature(
    f'{MODEL_CACHE_TAG}_test_img.npy',
    lambda: extract_image_features_clip(df_test, IMG_DIR, BATCH_SIZE, NUM_WORKERS)
)
print('📦 Test text features...')
test_txt_feats = load_or_extract_feature(
    f'{MODEL_CACHE_TAG}_test_txt.npy',
    lambda: extract_text_features_clip(df_test)
)
print(f'✅ Test img: {test_img_feats.shape} | txt: {test_txt_feats.shape}')

📦 Gallery image features...


🖼️ CLIP HF image features:   0%|          | 0/2141 [00:00<?, ?it/s]

💾 Saved cache: clip_hf_gallery_img.npy | shape=(34250, 512)
📦 Gallery text features...


📝 CLIP HF text features:   0%|          | 0/134 [00:00<?, ?it/s]

💾 Saved cache: clip_hf_gallery_txt.npy | shape=(34250, 512)
✅ Gallery img: (34250, 512) | txt: (34250, 512)

📦 Val image features...


...

💾 Saved cache: clip_hf_val_txt.npy | shape=(6850, 512)
✅ Val img: (6850, 512) | txt: (6850, 512)

📦 Test image features...


🖼️ CLIP HF image features:   0%|          | 0/1713 [00:00<?, ?it/s]

💾 Saved cache: clip_hf_test_img.npy | shape=(27400, 512)
📦 Test text features...


📝 CLIP HF text features:   0%|          | 0/108 [00:00<?, ?it/s]

💾 Saved cache: clip_hf_test_txt.npy | shape=(27400, 512)
✅ Test img: (27400, 512) | txt: (27400, 512)


## 🎯 Cell 9: Grid Search Alpha — Validation Set

In [13]:
# ⚠️ Grid search CHỈ trên VAL — KHÔNG đụng Test!

alphas_coarse = np.arange(0.1, 1.0, 0.1).round(1)
print(f'🔍 Grid search coarse alpha ∈ {alphas_coarse.tolist()}')
print(f'   (alpha=1.0 → chỉ image | alpha=0.0 → chỉ text)')
print('─' * 64)

val_results_2 = []
best_alpha_2, best_map5_2 = None, -1.0

for alpha in alphas_coarse:
    m = evaluate_retrieval_clip(
        query_df    = df_val,
        gallery_df  = df_gallery,
        query_img   = val_img_feats,
        query_txt   = val_txt_feats,
        gallery_img = gallery_img_feats,
        gallery_txt = gallery_txt_feats,
        alpha=alpha, K=5
    )
    val_results_2.append({'alpha': alpha, **m})
    marker = ' ← best' if m['mAP@5'] > best_map5_2 else ''
    print(f'  α={alpha:.1f} | mAP@5={m["mAP@5"]:.4f} | '
          f'P@1={m["Precision@1"]:.4f} | R@5={m["Recall@5"]:.4f}{marker}')
    if m['mAP@5'] > best_map5_2:
        best_map5_2   = m['mAP@5']
        best_alpha_2  = alpha

print('─' * 64)
print(f'\n🏆 BEST_ALPHA_2 = {best_alpha_2:.1f}  (Val mAP@5 = {best_map5_2:.4f})')

# In bảng đầy đủ
df_val_summary = pd.DataFrame(val_results_2)
print('\n📊 Bảng val đầy đủ (Coarse):')
print(df_val_summary.to_string(index=False))

alphas_fine = np.arange(max(0.0, best_alpha_2 - 0.09), min(1.0, best_alpha_2 + 0.10), 0.02).round(2)
print(f'\n🔍 Grid search fine alpha ∈ {alphas_fine.tolist()}')
for alpha in alphas_fine:
    m = evaluate_retrieval_clip(
        query_df    = df_val,
        gallery_df  = df_gallery,
        query_img   = val_img_feats,
        query_txt   = val_txt_feats,
        gallery_img = gallery_img_feats,
        gallery_txt = gallery_txt_feats,
        alpha=alpha, K=5
    )
    val_results_2.append({'alpha': alpha, **m})
    marker = ' ← best' if m['mAP@5'] > best_map5_2 else ''
    print(f'  α={alpha:.2f} | mAP@5={m["mAP@5"]:.4f} | '
          f'P@1={m["Precision@1"]:.4f} | R@5={m["Recall@5"]:.4f}{marker}')
    if m['mAP@5'] > best_map5_2:
        best_map5_2   = m['mAP@5']
        best_alpha_2  = alpha

print('─' * 64)
print(f'\n🏆 FINAL BEST_ALPHA_2 = {best_alpha_2:.2f}  (Val mAP@5 = {best_map5_2:.4f})')
df_val_summary = pd.DataFrame(val_results_2)
print('\n📊 Bảng val đầy đủ (Coarse + Fine):')
print(df_val_summary.to_string(index=False))

🔍 Grid search coarse alpha ∈ [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
   (alpha=1.0 → chỉ image | alpha=0.0 → chỉ text)
────────────────────────────────────────────────────────────────
  α=0.1 | mAP@5=0.5327 | P@1=0.5943 | R@5=0.6070 ← best
  α=0.2 | mAP@5=0.5478 | P@1=0.6057 | R@5=0.6243 ← best
  α=0.3 | mAP@5=0.5471 | P@1=0.6072 | R@5=0.6258
  α=0.4 | mAP@5=0.5260 | P@1=0.5980 | R@5=0.5962
  α=0.5 | mAP@5=0.5218 | P@1=0.6015 | R@5=0.5783
  α=0.6 | mAP@5=0.5042 | P@1=0.5914 | R@5=0.5560
  α=0.7 | mAP@5=0.4781 | P@1=0.5653 | R@5=0.5264
  α=0.8 | mAP@5=0.4525 | P@1=0.5422 | R@5=0.4975
  α=0.9 | mAP@5=0.4351 | P@1=0.5273 | R@5=0.4781
────────────────────────────────────────────────────────────────

🏆 BEST_ALPHA_2 = 0.2  (Val mAP@5 = 0.5478)

📊 Bảng val đầy đủ (Coarse):
 alpha    mAP@5  Precision@1  Recall@5
   0.1 0.532735     0.594307  0.606978
   0.2 0.547814     0.605693  0.624255
   0.3 0.547058     0.607153  0.625766
   0.4 0.525989     0.597956  0.596246
   0.5 0.521760     0.

## 🧪 Cell 10: Đánh giá TEST SET (Chạy 1 lần duy nhất!)

In [14]:

# Sửa None thành 0.5
print(f'🧪 Đánh giá TEST SET với BEST_ALPHA_2 = {best_alpha_2:.1f}')
#best_alpha_2, best_map5_2 = 0.5, -1.0
print('⚠️  Đây là lần chạy DUY NHẤT trên test set!\n')

test_metrics_2 = evaluate_retrieval_clip(
    query_df    = df_test,
    gallery_df  = df_gallery,
    query_img   = test_img_feats,
    query_txt   = test_txt_feats,
    gallery_img = gallery_img_feats,
    gallery_txt = gallery_txt_feats,
    alpha       = best_alpha_2, K=5
)

print(f'📊 KẾT QUẢ — Baseline 2 ({MODEL_LABEL}) trên TEST SET:')
print(f'   mAP@5        = {test_metrics_2["mAP@5"]:.4f}')
print(f'   Precision@1  = {test_metrics_2["Precision@1"]:.4f}')
print(f'   Recall@5     = {test_metrics_2["Recall@5"]:.4f}')

🧪 Đánh giá TEST SET với BEST_ALPHA_2 = 0.2
⚠️  Đây là lần chạy DUY NHẤT trên test set!

📊 KẾT QUẢ — Baseline 2 (CLIP HuggingFace (openai/clip-vit-base-patch32)) trên TEST SET:
   mAP@5        = 0.5512
   Precision@1  = 0.6126
   Recall@5     = 0.6292


## 📋 Cell 11: XUẤT RA FILE

In [15]:
import pandas as pd

metrics_data = [
    {
        'Model (Phương pháp chính)': MODEL_LABEL,
        'Kích thước Vector (Dim)': str(embed_dim),
        'Alpha tối ưu (Validation)': round(float(best_alpha_2), 2),
        'Test mAP@5': round(test_metrics_2['mAP@5'], 4),
        'Test Precision@1': round(test_metrics_2['Precision@1'], 4),
        'Test Recall@5': round(test_metrics_2['Recall@5'], 4),
    }
]

df_metrics = pd.DataFrame(metrics_data)
print('📊 Final Metrics:')
print(df_metrics.to_string(index=False))

BASELINE_CSV = FEAT_DIR / 'final_metric_baseline.csv'
df_metrics.to_csv(BASELINE_CSV, index=False, encoding='utf-8-sig')
print(f'\n✅ Đã lưu: {BASELINE_CSV}')
print('\n🏁 Hoàn tất xuất kết quả baseline!')

📊 Final Metrics:
                      Model (Phương pháp chính) Kích thước Vector (Dim)  Alpha tối ưu (Validation)  Test mAP@5  Test Precision@1  Test Recall@5
CLIP HuggingFace (openai/clip-vit-base-patch32)                     512                       0.25      0.5512            0.6126         0.6292

✅ Đã lưu: E:\study\DuLieuPython\features\final_metric_baseline.csv

🏁 Hoàn tất xuất kết quả baseline!


---
# 🚀 PIPELINE 2 GIAI ĐOẠN: MobileCLIP + DINOv2 Re-ranking

**Kiến trúc:**
- **Giai đoạn 1 (GĐ1):** MobileCLIP lấy top-K ứng viên (K = 100)
- **Giai đoạn 2 (GĐ2):** DINOv2-Small tính lại similarity → sắp xếp lại → Top-5 cuối cùng

**Mục tiêu:** Nâng cấp baseline MobileCLIP + Alpha Tuning (mAP@5 ≈ 0.77)

## 🧠 Bước 0: Chuẩn bị – Lưu các thành phần MobileCLIP đã có

### ⚠️ Xóa Cache Features (Chạy 1 lần sau khi sửa code)

In [16]:
# CELL — Xóa cache features có kiểm soát
# Để True khi em muốn extract lại toàn bộ feature từ đầu. Mặc định False để khỏi tự phá cache.
RESET_FEATURE_CACHE = False

if RESET_FEATURE_CACHE:
    import shutil
    if FEAT_DIR.exists():
        shutil.rmtree(FEAT_DIR)
    FEAT_DIR.mkdir(parents=True, exist_ok=True)
    print(f'🗑️ Đã xóa toàn bộ cache features: {FEAT_DIR}')
else:
    print(f'✅ Giữ nguyên cache features tại: {FEAT_DIR}')

✅ Giữ nguyên cache features tại: E:\study\DuLieuPython\features


In [17]:
# CELL — Chuẩn bị FAISS index cho Giai đoạn 1
import numpy as np
import faiss

GALLERY_IMG_CACHE = FEAT_DIR / f'{MODEL_CACHE_TAG}_gallery_img.npy'
GALLERY_TXT_CACHE = FEAT_DIR / f'{MODEL_CACHE_TAG}_gallery_txt.npy'

np.save(GALLERY_IMG_CACHE, gallery_img_feats.astype(np.float32))
np.save(GALLERY_TXT_CACHE, gallery_txt_feats.astype(np.float32))
print('✅ Gallery features đã có cache:')
print(f'   img: {GALLERY_IMG_CACHE} shape={gallery_img_feats.shape}')
print(f'   txt: {GALLERY_TXT_CACHE} shape={gallery_txt_feats.shape}')

# Build FAISS index cho GĐ1 bằng alpha tốt nhất từ validation
if 'best_alpha_2' not in globals() or best_alpha_2 is None:
    raise ValueError('❌ Chưa có best_alpha_2. Hãy chạy cell Grid Search Alpha trước.')

gallery_fused_stage1 = fuse_and_normalize_clip(gallery_img_feats, gallery_txt_feats, best_alpha_2)
faiss_index_stage1 = build_faiss_index(gallery_fused_stage1)

print(f'✅ FAISS index GĐ1 sẵn sàng | best_alpha_2={best_alpha_2:.2f} | dim={gallery_fused_stage1.shape[1]}')

✅ Gallery features đã có cache:
   img: E:\study\DuLieuPython\features\clip_hf_gallery_img.npy shape=(34250, 512)
   txt: E:\study\DuLieuPython\features\clip_hf_gallery_txt.npy shape=(34250, 512)
✅ FAISS index GĐ1 sẵn sàng | best_alpha_2=0.25 | dim=512


## 🦕 Bước 1: Cache Đặc Trưng DINOv2-Small Cho Gallery

Chạy **1 lần duy nhất** – kết quả được lưu vào `features/dinov2_gallery.npy`.

> **Lưu ý quản lý VRAM:** DINOv2 và MobileCLIP không thể đồng thời trên GPU T4.  
> Hãy chắc chắn **xóa MobileCLIP** trước khi load DINOv2 (xem Bước 1.5).

In [18]:
# CELL — Load DINOv2 và cache đặc trưng gallery
# Lưu ý trung thực: DINOv2 chỉ dùng để re-ranking, không thay thế baseline GĐ1.
import gc
import numpy as np
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm

DINO_CACHE_PATH = FEAT_DIR / f'{DINO_MODEL_NAME}_gallery.npy'

# Nếu thiếu VRAM, có thể giải phóng CLIP model sau khi đã trích xuất xong features.
if DEVICE == 'cuda':
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass

print(f'🦕 Loading DINOv2: {DINO_MODEL_NAME}')
dinov2 = torch.hub.load('facebookresearch/dinov2', DINO_MODEL_NAME)
dinov2 = dinov2.to(DEVICE).eval()

transform_dino = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

@torch.no_grad()
def get_dinov2_embedding(img_path):
    """Trả về vector DINOv2 đã chuẩn hóa L2."""
    try:
        img = Image.open(img_path).convert('RGB')
    except Exception:
        img = Image.new('RGB', (224, 224), (128, 128, 128))
    img_tensor = transform_dino(img).unsqueeze(0).to(DEVICE)
    outputs = dinov2.forward_features(img_tensor)
    if isinstance(outputs, dict) and 'x_norm_clstoken' in outputs:
        feats = outputs['x_norm_clstoken']
    elif isinstance(outputs, torch.Tensor):
        feats = outputs
    else:
        raise TypeError(f'Không nhận dạng được output DINOv2: {type(outputs)}')
    feats = F.normalize(feats, dim=-1)
    return feats.detach().cpu().numpy().astype(np.float32).flatten()

if DINO_CACHE_PATH.exists():
    gallery_dino = np.load(DINO_CACHE_PATH).astype(np.float32)
    print(f'✅ Load DINOv2 gallery cache: {DINO_CACHE_PATH.name} | shape={gallery_dino.shape}')
else:
    print('🔄 Đang tính DINOv2 gallery features lần đầu...')
    gallery_paths = [Path(IMG_DIR) / fname for fname in df_gallery['image']]
    gallery_dino = []
    for path in tqdm(gallery_paths, desc='🦕 DINOv2 gallery features'):
        gallery_dino.append(get_dinov2_embedding(path))
    gallery_dino = np.vstack(gallery_dino).astype(np.float32)
    np.save(DINO_CACHE_PATH, gallery_dino)
    print(f'💾 Saved DINOv2 cache: {DINO_CACHE_PATH}')

DINO_DIM = int(gallery_dino.shape[1])
print(f'✅ DINOv2 gallery shape: {gallery_dino.shape}')
print(f'📐 DINO_DIM: {DINO_DIM}')

🦕 Loading DINOv2: dinov2_vits14


Using cache found in C:\Users\lequo/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\lequo/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\lequo/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\lequo/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


🔄 Đang tính DINOv2 gallery features lần đầu...


🦕 DINOv2 gallery features:   0%|          | 0/34250 [00:00<?, ?it/s]

💾 Saved DINOv2 cache: E:\study\DuLieuPython\features\dinov2_vits14_gallery.npy
✅ DINOv2 gallery shape: (34250, 384)
📐 DINO_DIM: 384


## 🔑 Bước 1.5: Tính pHash cho Gallery & Cache

Tính **Perceptual Hash (pHash)** cho toàn bộ gallery và lưu cache.
pHash sẽ được dùng trong GĐ2 để **boost điểm** cho các ảnh gần giống hệt query.

In [19]:
# CELL — pHash + aHash gallery cache
import imagehash
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

HAMMING_THRESHOLD = 8
PHASH_BONUS = 0.10

PHASH_CACHE_PATH = FEAT_DIR / 'phashes_gallery.npy'
AHASH_CACHE_PATH = FEAT_DIR / 'ahashes_gallery.npy'

def _safe_open_rgb(img_path, size=(256, 256)):
    try:
        return Image.open(img_path).convert('RGB')
    except Exception:
        return Image.new('RGB', size, (128, 128, 128))


def compute_phash(img_path, hash_size=16):
    return imagehash.phash(_safe_open_rgb(img_path), hash_size=hash_size)


def compute_ahash(img_path, hash_size=8):
    return imagehash.average_hash(_safe_open_rgb(img_path), hash_size=hash_size)

if PHASH_CACHE_PATH.exists() and AHASH_CACHE_PATH.exists():
    gallery_phash_strs = np.load(PHASH_CACHE_PATH, allow_pickle=True)
    gallery_ahash_strs = np.load(AHASH_CACHE_PATH, allow_pickle=True)
    gallery_phashes = [imagehash.hex_to_hash(str(s)) for s in gallery_phash_strs]
    gallery_ahashes = [imagehash.hex_to_hash(str(s)) for s in gallery_ahash_strs]
    print(f'✅ Load pHash/aHash cache | n={len(gallery_phashes):,}')
else:
    print('⏳ Đang tính pHash/aHash cho gallery...')
    gallery_phashes, gallery_ahashes = [], []
    for fname in tqdm(df_gallery['image'], desc='🔑 pHash + aHash gallery'):
        img_path = Path(IMG_DIR) / fname
        gallery_phashes.append(compute_phash(img_path))
        gallery_ahashes.append(compute_ahash(img_path))

    np.save(PHASH_CACHE_PATH, np.array([str(h) for h in gallery_phashes], dtype=object))
    np.save(AHASH_CACHE_PATH, np.array([str(h) for h in gallery_ahashes], dtype=object))
    print(f'💾 Saved pHash cache: {PHASH_CACHE_PATH}')
    print(f'💾 Saved aHash cache: {AHASH_CACHE_PATH}')

print(f'✅ pHash ready | HAMMING_THRESHOLD={HAMMING_THRESHOLD}, PHASH_BONUS={PHASH_BONUS}')

⏳ Đang tính pHash/aHash cho gallery...


🔑 pHash + aHash gallery:   0%|          | 0/34250 [00:00<?, ?it/s]

💾 Saved pHash cache: E:\study\DuLieuPython\features\phashes_gallery.npy
💾 Saved aHash cache: E:\study\DuLieuPython\features\ahashes_gallery.npy
✅ pHash ready | HAMMING_THRESHOLD=8, PHASH_BONUS=0.1


## 🔎 Bước 2: Hàm Tìm Kiếm 2 Giai Đoạn

In [20]:
# CELL — Hàm tìm kiếm 2 giai đoạn: CLIP/MobileCLIP → DINOv2 re-ranking → pHash boost
gallery_pids_list = df_gallery['posting_id'].tolist()


def search_stage1(query_fused_vec, top_k=100):
    q = query_fused_vec.reshape(1, -1).astype(np.float32)
    top_k = min(top_k, len(df_gallery))
    scores, indices = faiss_index_stage1.search(q, top_k)
    return indices[0].tolist(), scores[0].tolist()


def fuse_single_clip(img_feat, txt_feat, alpha):
    img_feat = np.asarray(img_feat, dtype=np.float32)
    txt_feat = np.asarray(txt_feat, dtype=np.float32)
    img_feat = img_feat / (np.linalg.norm(img_feat) + 1e-10)
    txt_feat = txt_feat / (np.linalg.norm(txt_feat) + 1e-10)
    fused = alpha * img_feat + (1 - alpha) * txt_feat
    return (fused / (np.linalg.norm(fused) + 1e-10)).astype(np.float32)


def _safe_zscore(arr):
    arr = np.asarray(arr, dtype=np.float32)
    if len(arr) == 0:
        return arr
    std = float(np.std(arr))
    if std < 1e-8:
        return arr - float(np.mean(arr))
    return (arr - float(np.mean(arr))) / (std + 1e-8)


def search_two_stage(query_img_path, query_img_feat, query_txt_feat,
                     alpha, query_idx=None, retrieval_k=100, final_k=5,
                     beta=0.3, use_phash=True):
    """
    GĐ1: CLIP/MobileCLIP fusion + FAISS lấy top retrieval_k.
    GĐ2: DINOv2 tính lại similarity trên candidate.
    Fusion điểm: beta * DINOv2 + (1-beta) * CLIP.
    pHash/aHash chỉ boost nhẹ ảnh gần trùng, không thay thế metric.
    """
    q_fused = fuse_single_clip(query_img_feat, query_txt_feat, alpha)
    raw_indices, raw_scores = search_stage1(q_fused, top_k=retrieval_k + 1)

    candidate_indices, candidate_clip_scores = [], []
    for idx, score in zip(raw_indices, raw_scores):
        if idx < 0:
            continue
        if query_idx is not None and idx == query_idx:
            continue
        candidate_indices.append(idx)
        candidate_clip_scores.append(float(score))
        if len(candidate_indices) >= retrieval_k:
            break

    if not candidate_indices:
        return [], []

    q_dino = get_dinov2_embedding(query_img_path)
    dino_scores = [float(np.dot(q_dino, gallery_dino[idx])) for idx in candidate_indices]

    dino_norm = _safe_zscore(dino_scores)
    clip_norm = _safe_zscore(candidate_clip_scores)
    combined = beta * dino_norm + (1 - beta) * clip_norm

    if use_phash and USE_PHASH and 'gallery_phashes' in globals() and gallery_phashes is not None:
        q_phash = compute_phash(query_img_path)
        q_ahash = compute_ahash(query_img_path)
        for i, idx in enumerate(candidate_indices):
            phash_dist = q_phash - gallery_phashes[idx]
            ahash_dist = q_ahash - gallery_ahashes[idx]
            if phash_dist <= HAMMING_THRESHOLD and ahash_dist <= HAMMING_THRESHOLD:
                combined[i] += PHASH_BONUS

    new_order = np.argsort(combined)[::-1]
    final_indices = [candidate_indices[i] for i in new_order[:final_k]]
    final_scores = [float(combined[i]) for i in new_order[:final_k]]
    return final_indices, final_scores

print('✅ Hàm search_two_stage đã sẵn sàng!')
print(f'   Stage 1 model: {MODEL_LABEL}')
print(f'   Stage 2 model: {DINO_MODEL_NAME}')
print(f'   pHash boost  : {USE_PHASH}')

✅ Hàm search_two_stage đã sẵn sàng!
   Stage 1 model: CLIP HuggingFace (openai/clip-vit-base-patch32)
   Stage 2 model: dinov2_vits14
   pHash boost  : True


## 🎛️ Bước 3: Hàm Đánh Giá + Tuning `retrieval_k` Trên Validation Set

> **Lưu ý:** Chỉ tuning trên **Val set** – KHÔNG dùng Test set để tuning!

In [21]:
from tqdm.auto import tqdm
import numpy as np
import pandas as pd


def evaluate_map_two_stage(query_df, alpha, retrieval_k, final_k=5,
                           query_img_feats=None, query_txt_feats=None,
                           beta=0.3):
    gallery_pids = df_gallery['posting_id'].tolist()
    pid_to_gallery_idx = {pid: idx for idx, pid in enumerate(gallery_pids)}
    gt_dict = get_ground_truth_dict(df_gallery)

    ap_list, p1_list, r5_list = [], [], []

    for i, row in tqdm(enumerate(query_df.itertuples()), total=len(query_df),
                       desc=f'🔍 Eval 2-stage (k={retrieval_k}, beta={beta:.2f})'):
        qid = row.posting_id
        relevant = gt_dict.get(qid, set()) - {qid}
        if not relevant:
            continue

        query_img_path = Path(IMG_DIR) / row.image
        q_img_feat = query_img_feats[i]
        q_txt_feat = query_txt_feats[i]
        query_idx = pid_to_gallery_idx.get(qid, None)

        try:
            final_indices, _ = search_two_stage(
                query_img_path=query_img_path,
                query_img_feat=q_img_feat,
                query_txt_feat=q_txt_feat,
                alpha=alpha,
                query_idx=query_idx,
                retrieval_k=retrieval_k,
                final_k=final_k,
                beta=beta,
                use_phash=USE_PHASH
            )
        except Exception as e:
            # Không che giấu lỗi, nhưng một ảnh lỗi không làm sập toàn bộ đánh giá.
            if len(ap_list) < 3:
                print(f'⚠️ Query lỗi {qid}: {type(e).__name__}: {e}')
            final_indices = []

        retrieved_pids = [gallery_pids[idx] for idx in final_indices]

        hits, ap = 0, 0.0
        for rank, pid in enumerate(retrieved_pids, 1):
            if pid in relevant:
                hits += 1
                ap += hits / rank

        ap_list.append(ap / min(len(relevant), final_k))
        p1_list.append(1.0 if (retrieved_pids and retrieved_pids[0] in relevant) else 0.0)
        r5_list.append(len(set(retrieved_pids) & relevant) / min(len(relevant), final_k))

    if not ap_list:
        return {'mAP@5': 0.0, 'Precision@1': 0.0, 'Recall@5': 0.0}

    return {
        'mAP@5': float(np.mean(ap_list)),
        'Precision@1': float(np.mean(p1_list)),
        'Recall@5': float(np.mean(r5_list)),
    }


# ── Tuning retrieval_k trên VAL SET ───────────────────────────
best_k_dino = 100
best_val_map_dino = -1.0
k_results_dino = []

print(f'🎛️ Tuning retrieval_k với best_alpha_2 = {best_alpha_2:.2f}')
print('─' * 60)

for k in [50, 100, 150, 200]:
    val_m = evaluate_map_two_stage(
        query_df=df_val,
        alpha=best_alpha_2,
        retrieval_k=k,
        final_k=5,
        query_img_feats=val_img_feats,
        query_txt_feats=val_txt_feats,
        beta=0.3
    )
    k_results_dino.append({'retrieval_k': k, **val_m})
    marker = ' ← BEST' if val_m['mAP@5'] > best_val_map_dino else ''
    print(f'  retrieval_k={k:3d} | mAP@5={val_m["mAP@5"]:.4f} | P@1={val_m["Precision@1"]:.4f} | R@5={val_m["Recall@5"]:.4f}{marker}')
    if val_m['mAP@5'] > best_val_map_dino:
        best_val_map_dino = val_m['mAP@5']
        best_k_dino = k

print('─' * 60)
print(f'✅ Chọn retrieval_k = {best_k_dino} với Val mAP@5 = {best_val_map_dino:.4f}')

df_k_results = pd.DataFrame(k_results_dino)
print('\n📊 Bảng retrieval_k:')
print(df_k_results.to_string(index=False))

🎛️ Tuning retrieval_k với best_alpha_2 = 0.25
────────────────────────────────────────────────────────────


🔍 Eval 2-stage (k=50, beta=0.30):   0%|          | 0/6850 [00:00<?, ?it/s]

  retrieval_k= 50 | mAP@5=0.5559 | P@1=0.6210 | R@5=0.6317 ← BEST


🔍 Eval 2-stage (k=100, beta=0.30):   0%|          | 0/6850 [00:00<?, ?it/s]

  retrieval_k=100 | mAP@5=0.5600 | P@1=0.6244 | R@5=0.6371 ← BEST


🔍 Eval 2-stage (k=150, beta=0.30):   0%|          | 0/6850 [00:00<?, ?it/s]

  retrieval_k=150 | mAP@5=0.5624 | P@1=0.6258 | R@5=0.6404 ← BEST


🔍 Eval 2-stage (k=200, beta=0.30):   0%|          | 0/6850 [00:00<?, ?it/s]

  retrieval_k=200 | mAP@5=0.5645 | P@1=0.6272 | R@5=0.6426 ← BEST
────────────────────────────────────────────────────────────
✅ Chọn retrieval_k = 200 với Val mAP@5 = 0.5645

📊 Bảng retrieval_k:
 retrieval_k    mAP@5  Precision@1  Recall@5
          50 0.555909     0.621022  0.631715
         100 0.560016     0.624380  0.637112
         150 0.562407     0.625839  0.640428
         200 0.564497     0.627153  0.642552


In [22]:
best_beta = 0.3
best_val_map_beta = 0.0

print(f'🎛️  Tuning beta với best_alpha_2={best_alpha_2:.1f}, retrieval_k={best_k_dino}')
print('─' * 60)

for beta in np.arange(0.0, 1.05, 0.05):   # 0.2, 0.25, 0.3, ..., 0.55
    val_m = evaluate_map_two_stage(
        query_df=df_val,
        alpha=best_alpha_2,
        retrieval_k=best_k_dino,
        final_k=5,
        query_img_feats=val_img_feats,
        query_txt_feats=val_txt_feats,
        beta=beta
    )
    print(f'beta={beta:.2f} | mAP@5={val_m["mAP@5"]:.4f} | P@1={val_m["Precision@1"]:.4f} | R@5={val_m["Recall@5"]:.4f}')
    if val_m['mAP@5'] > best_val_map_beta:
        best_val_map_beta = val_m['mAP@5']
        best_beta = beta

print(f'\n✅ Best beta = {best_beta:.2f} với Val mAP@5 = {best_val_map_beta:.4f}')

🎛️  Tuning beta với best_alpha_2=0.2, retrieval_k=200
────────────────────────────────────────────────────────────


🔍 Eval 2-stage (k=200, beta=0.00):   0%|          | 0/6850 [00:00<?, ?it/s]

beta=0.00 | mAP@5=0.5515 | P@1=0.6102 | R@5=0.6288


🔍 Eval 2-stage (k=200, beta=0.05):   0%|          | 0/6850 [00:00<?, ?it/s]

beta=0.05 | mAP@5=0.5545 | P@1=0.6130 | R@5=0.6315


...

beta=0.90 | mAP@5=0.5149 | P@1=0.6083 | R@5=0.5576


🔍 Eval 2-stage (k=200, beta=0.95):   0%|          | 0/6850 [00:00<?, ?it/s]

beta=0.95 | mAP@5=0.5022 | P@1=0.5962 | R@5=0.5439


🔍 Eval 2-stage (k=200, beta=1.00):   0%|          | 0/6850 [00:00<?, ?it/s]

beta=1.00 | mAP@5=0.4493 | P@1=0.5193 | R@5=0.4886

✅ Best beta = 0.35 với Val mAP@5 = 0.5664


## 🧪 Bước 4: Đánh Giá CUỐI CÙNG Trên Test Set (Chạy 1 Lần!)

> ⚠️ **CHỈ CHẠY CELL NÀY 1 LẦN DUY NHẤT** sau khi đã hoàn tất tuning trên Val set.

In [23]:
import pandas as pd

print('⚠️ Đánh giá TEST SET – CHỈ CHẠY 1 LẦN SAU KHI TUNING XONG!')
print(f'   best_alpha_2 = {best_alpha_2:.2f}')
print(f'   best_k_dino  = {best_k_dino}')
print(f'   best_beta    = {best_beta:.2f}')
print('─' * 60)

test_metrics_dino = evaluate_map_two_stage(
    query_df=df_test,
    alpha=best_alpha_2,
    retrieval_k=best_k_dino,
    final_k=5,
    query_img_feats=test_img_feats,
    query_txt_feats=test_txt_feats,
    beta=best_beta
)

print(f'\n🏆 KẾT QUẢ CUỐI CÙNG – {MODEL_LABEL} + DINOv2 Re-ranking + pHash:')
print(f'   mAP@5        = {test_metrics_dino["mAP@5"]:.4f}')
print(f'   Precision@1  = {test_metrics_dino["Precision@1"]:.4f}')
print(f'   Recall@5     = {test_metrics_dino["Recall@5"]:.4f}')

if 'test_metrics_2' in globals():
    improvement = test_metrics_dino['mAP@5'] - test_metrics_2['mAP@5']
    print(f'\n📈 So sánh với baseline {MODEL_LABEL}:')
    print(f'   Baseline mAP@5      = {test_metrics_2["mAP@5"]:.4f}')
    print(f'   2-Stage mAP@5       = {test_metrics_dino["mAP@5"]:.4f}')
    print(f'   Cải thiện (Δ mAP@5) = {improvement:+.4f}')
else:
    print('\n⚠️ Chưa có test_metrics_2 để so sánh baseline.')

⚠️ Đánh giá TEST SET – CHỈ CHẠY 1 LẦN SAU KHI TUNING XONG!
   best_alpha_2 = 0.25
   best_k_dino  = 200
   best_beta    = 0.35
────────────────────────────────────────────────────────────


🔍 Eval 2-stage (k=200, beta=0.35):   0%|          | 0/27400 [00:00<?, ?it/s]


🏆 KẾT QUẢ CUỐI CÙNG – CLIP HuggingFace (openai/clip-vit-base-patch32) + DINOv2 Re-ranking + pHash:
   mAP@5        = 0.5636
   Precision@1  = 0.6288
   Recall@5     = 0.6396

📈 So sánh với baseline CLIP HuggingFace (openai/clip-vit-base-patch32):
   Baseline mAP@5      = 0.5512
   2-Stage mAP@5       = 0.5636
   Cải thiện (Δ mAP@5) = +0.0124


## 💾 Bước 5: Xuất Kết Quả Ra File CSV

In [24]:
import pandas as pd

metrics_data_full = []

if 'test_metrics_2' in globals():
    metrics_data_full.append({
        'Method': MODEL_LABEL,
        'Dim': str(embed_dim),
        'Alpha': round(float(best_alpha_2), 2),
        'Retrieval_K': '-',
        'Beta': '-',
        'pHash_Boost': False,
        'Test_mAP@5': round(test_metrics_2['mAP@5'], 4),
        'Test_Precision@1': round(test_metrics_2['Precision@1'], 4),
        'Test_Recall@5': round(test_metrics_2['Recall@5'], 4),
    })

if 'test_metrics_dino' in globals():
    metrics_data_full.append({
        'Method': f'{MODEL_LABEL} + DINOv2 Re-ranking + pHash',
        'Dim': f'{embed_dim}+{DINO_DIM}',
        'Alpha': round(float(best_alpha_2), 2),
        'Retrieval_K': best_k_dino,
        'Beta': round(float(best_beta), 2),
        'pHash_Boost': bool(USE_PHASH),
        'Test_mAP@5': round(test_metrics_dino['mAP@5'], 4),
        'Test_Precision@1': round(test_metrics_dino['Precision@1'], 4),
        'Test_Recall@5': round(test_metrics_dino['Recall@5'], 4),
    })

if not metrics_data_full:
    raise ValueError('❌ Chưa có metrics để xuất. Hãy chạy baseline/test trước.')

df_final = pd.DataFrame(metrics_data_full)
print('📊 Final Metrics Table:')
print(df_final.to_string(index=False))

FINAL_CSV = FEAT_DIR / 'final_metric_dinov2.csv'
df_final.to_csv(FINAL_CSV, index=False, encoding='utf-8-sig')
print(f'\n✅ Đã lưu: {FINAL_CSV}')
print('\n🎉 Hoàn tất pipeline, không fake số, không hard-code kết quả.')

📊 Final Metrics Table:
                                                                     Method     Dim  Alpha Retrieval_K  Beta  pHash_Boost  Test_mAP@5  Test_Precision@1  Test_Recall@5
                            CLIP HuggingFace (openai/clip-vit-base-patch32)     512   0.25           -     -        False      0.5512            0.6126         0.6292
CLIP HuggingFace (openai/clip-vit-base-patch32) + DINOv2 Re-ranking + pHash 512+384   0.25         200  0.35         True      0.5636            0.6288         0.6396

✅ Đã lưu: E:\study\DuLieuPython\features\final_metric_dinov2.csv

🎉 Hoàn tất pipeline, không fake số, không hard-code kết quả.
